# Optimizing Agentic AI Collaboration v2.4.2 — Gemma-4-E4B-it Single-Agent Baseline
## MMLU only

This notebook is the MMLU companion to `v2.4.0` (*Gemma-4-E4B-it Single-Agent Baseline GSM8K*): same `RUN_NAME` / `CONFIG` / `GEN`, so its W&B run and artifact land alongside the `v2.4.0` GSM8K run under the same baseline name. GSM8K is **not** run here — that result lives in `v2.4.0`.

Splitting the two benchmarks across separate notebooks is a **deliberate scoping choice**: Kaggle enforces a 12-hour session limit, and evaluating both GSM8K and MMLU under 3 generation seeds for this model does not reliably fit within that window. This is not error recovery — each notebook is designed from the outset to run a single benchmark per Kaggle session.

**Model:** `google/gemma-4-E4B-it` (multimodal checkpoint, used **text-only**) with 4-bit NF4 quantization via bitsandbytes — the same load-time quantization workflow as the other SLM families in this project.
**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier).

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `HF_TOKEN` — Hugging Face read-access token
   - `WANDB_API_KEY` — Weights & Biases API key
3. Click **Run All**. The first cell auto-detects Kaggle, clones the repo, and loads secrets.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## What this notebook implements

A **single-agent baseline** over **MMLU** with W&B tracking — identical methodology to `v2.4.0`, applied to the other benchmark:

* A single direct-inference call per MMLU question (no Proposer/Critic/Refiner pipeline).
* **W&B logging** of per-call latency, output/total token counts, GPU peak VRAM, and generation throughput; plus a unified W&B `Table` of every sample.
* Each question is run under **3 generation seeds**; correctness, latency, token and peak-VRAM matrices (N×m) are persisted to a W&B artifact for downstream analysis.
* **Accuracy, latency, output/total tokens and peak VRAM** are summarized with a point estimate **and a cluster-robust 95% confidence interval**.

This produces the MMLU half of the Gemma-4-E4B-it single-agent baseline (the GSM8K half is `v2.4.0`). Overhead ratios/deltas are **not** computed here — that remains a MAS-vs-baseline comparison for later notebooks.

## Relation with the project WPs

**WP2 - T3: Single-agent baseline establishment (CoT).** This notebook establishes the zero-shot, CoT, single-agent baseline for one of the discuted state-of-the-art SLMs (Gemma-4-E4B) on MMLU. This baseline is the reference the L1/L2 multi-agent protocols (WP2 - T4/T6) must overcome, and its latency/token/VRAM measurements feed the WP3 - T1 compute-accuracy trade-off analysis.


## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `datasets`, `wandb`), all pinned in `requirements.txt`. If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [ ]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [2]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

/kaggle/working/tfg-agentic-ai
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: kaggle
✅ Secrets loaded from Kaggle Secrets.


## 3. Define Runs Metadata

We define the W&B project and configuration metadata, in order to later create a W&B run that will store all the baseline calls, metrics and statistics of the model's execution.

In [ ]:
RUN_NAME = "Baseline-Gemma-4-E4B-it"

PROJECT = "agentic-final-results" if (ENVIRONMENT == "kaggle") else "dev-agentic-baselines"
print(f"W&B Project: {PROJECT}")

GEN = {
    "max_new_tokens": 1024,
    "temperature":    0.7,
    "top_p":          0.95,
    "top_k":          20,
}

CONFIG = {
        "model_id":         "google/gemma-4-E4B-it",
        "quantization":     "nf4-4bit",
        "framework":        "langchain",
        "protocol":         "single_agent_baseline",
        "eval_size":        128,
        "data_seed":        789,
        "generation_seeds": [42, 123, 456],
    }

## 4. Data Loading
We load only the **MMLU** dataset — GSM8K is evaluated separately by `v2.4.0`, so there's no need to load or run it here.

In [5]:
from utils.data import load_mmlu

# ── MMLU DATASET ──────────────────────────────────────────────────
mmlu_dataset = load_mmlu(split="test")

print(f"✅ MMLU dataset loaded with {len(mmlu_dataset)} examples.")
print("Example Subject:", mmlu_dataset[0]['subject'])
print("Example Question:", mmlu_dataset[0]['question'])
print("Example Choices:", mmlu_dataset[0]['choices'])
print("Example Answer Index:", mmlu_dataset[0]['answer'])  # Returns an integer 0-3

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

✅ MMLU dataset loaded with 14042 examples.
Example Subject: abstract_algebra
Example Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Example Choices: ['0', '4', '2', '6']
Example Answer Index: 1


## 5. Model Loading (4-bit Quantized)

Gemma-4-E4B-it ships as a **multimodal checkpoint** (a language model plus vision and audio encoders), but this notebook uses it **text-only**. `load_model_and_tokenizer` dispatches Gemma-4 to a bespoke branch: it loads via `AutoModelForImageTextToText` and then **strips the vision/audio towers** — the model's `forward` gates every image/audio op behind the corresponding inputs, so text generation never invokes them, and dropping them reclaims that VRAM. The remaining language model is quantized to **NF4 4-bit** on the fly via bitsandbytes, keeping VRAM well within the T4's 16 GB — the same load-time quantization workflow used for every other SLM family here. The tokenizer is a standard HuggingFace fast tokenizer with a built-in instruct chat template — no `mistral-common` dependency, unlike Ministral-3.

In [ ]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = CONFIG["model_id"]
model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)

print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

## 6. Defining the Single-Agent Baseline

The baseline is a single LangChain chain: a `PromptTemplate` piped to the LLM (execution engine). No Proposer/Critic/Refiner — just one direct call per question. Only the **MMLU** template is needed here (letter-choice answer); the GSM8K template lives in `v2.4.0` and isn't used by this notebook.

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Parser
parser = StrOutputParser()

# ── MMLU Template ────────────────────────────────────
mmlu_template = PromptTemplate.from_template(
        "Your answer should be of the following format: 'ANSWER: [LETTER]' "
        "(without quotes) where [LETTER] is one of A,B,C,D. "
        "Think step by step before answering.\n\n"
        "{task}"
)

print("✅ MMLU Single-Agent chain template defined.")

✅ MMLU Single-Agent chain template defined.


## 7. Single-Agent Runner with W&B Logging

`run_agent_step` times a single `chain.invoke`, counts output/total tokens with the model's own tokenizer (outside the timed region so latency stays clean), and measures per-call peak VRAM. `run_single_agent_baseline` wraps one baseline call against the MMLU template, logging latency, tokens, peak VRAM and throughput per sample. **No overhead ratios are computed** — this is a baseline, not a MAS-vs-baseline comparison.

In [8]:
import time
from utils.logging_utils import log_metrics
from utils.instrumentation import (
    count_tokens, render_prompt,
    reset_gpu_peak, gpu_peak_mem_mb,
    tokens_per_second,
)

def run_agent_step(chain, inputs, step_name, prompt_template):
    """Run a single LangChain chain, time it, count tokens, measure per-step peak VRAM and throughput (tokens/s).

    Token counting and VRAM measurement are performed outside the timed region
    so the latency measurement (and the throughput one) wraps only ``chain.invoke``.

    Returns
    -------
    tuple[str, float, int, int, float]
        (output_text, elapsed_s, input_tokens, output_tokens, peak_vram_mb)
    """
    # Input tokens: counted BEFORE the timed region
    prompt_str   = render_prompt(prompt_template, inputs)
    input_tokens = count_tokens(tokenizer, prompt_str)

    # Reset peak VRAM counter for this step's independent measurement
    reset_gpu_peak()

    # Timed region: pure chain.invoke only
    t0          = time.perf_counter()
    output_text = chain.invoke(inputs)
    elapsed     = time.perf_counter() - t0

    # Post-invoke measurements: outside the timed region
    output_tokens = count_tokens(tokenizer, output_text)
    total_tokens  = input_tokens + output_tokens
    peak_vram_mb  = gpu_peak_mem_mb()
    tps           = tokens_per_second(output_tokens, elapsed)

    print(
        f"  ⏱  {step_name}: {elapsed:.1f}s  |  "
        f"{len(output_text)} chars  |  "
        f"in={input_tokens} out={output_tokens} tok  |  "
        f"peak={peak_vram_mb:.0f} MB  |  {tps:.1f} tok/s"
    )
    return output_text, elapsed, input_tokens, output_tokens, peak_vram_mb


def run_single_agent_baseline(chain, task, task_label="mmlu_sample"):
    """Execute a single-agent baseline call against the MMLU template, with W&B
    logging of latency, tokens, GPU peak VRAM and throughput. No multi-agent
    pipeline and no overhead metrics.

    Parameters
    ----------
    chain     : LangChain chain built for the current generation seed.
    task      : Formatted MMLU prompt string.
    task_label: W&B metric prefix (e.g. ``"mmlu_sample"``).

    Returns
    -------
    dict with keys:
        baseline                                   (text output)
        t_base                                     (latency, s)
        base_in_tok, base_out_tok, base_total_tok  (baseline tokens)
        base_peak_mb                               (peak VRAM, MB)
        base_throughput                            (tok/s)
    """
    print(f"\n{'='*60}")
    print(f"📊 SINGLE-AGENT BASELINE  —  {task_label}")
    print(f"{'='*60}")

    baseline, t_base, in_base, out_base, peak_base = run_agent_step(
        chain, {"task": task},
        f"{task_label}/baseline", mmlu_template
    )

    base_total_tok  = in_base + out_base
    base_throughput = tokens_per_second(out_base, t_base)

    # ── Log per-sample baseline summary to W&B (no overhead metrics) ──
    log_metrics({
        f"{task_label}/latency_s":      round(t_base, 2),
        f"{task_label}/output_tokens":  out_base,
        f"{task_label}/total_tokens":   base_total_tok,
        f"{task_label}/gpu_peak_mb":    round(peak_base, 2),
        f"{task_label}/throughput_tps": round(base_throughput, 2),
    })

    return {
        "baseline":        baseline,
        "t_base":          t_base,
        "base_in_tok":     in_base,
        "base_out_tok":    out_base,
        "base_total_tok":  base_total_tok,
        "base_peak_mb":    peak_base,
        "base_throughput": base_throughput,
    }

print("✅ Single-agent runner ready.")

✅ Single-agent runner ready.


## 8. Model Warmup
Before timing, we "warm up" the model with 2 dummy executions so the one-time CUDA / bitsandbytes kernel compilation and cuBLAS autotuning happen here — not inside the first timed sample.

In [9]:
# ── Warmup ────────────────────────────────────────────────────────────
# Run a couple of throwaway generations through the same pipeline the
# evaluation uses, so one-time CUDA / bitsandbytes kernel compilation and
# cuBLAS autotuning happen here — not inside the first timed sample.
from utils.warmup import warmup_model

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN["temperature"],
    top_p=GEN["top_p"], top_k=GEN["top_k"],
)

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'top_k', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔥 Warming up model (2 iterations, max_new_tokens=256)…


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Warmup complete in 20.1s — first evaluated sample will no longer pay cold-start cost.


20.051499358

## 9. MMLU — Initialize Weights & Biases

We open a dedicated W&B run for the MMLU evaluation, using the same `RUN_NAME` as `v2.4.0` so this run sits alongside the original GSM8K run under the same baseline name in W&B. The run is closed at the end of section 11.

In [10]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics

run = init_wandb(
    run_name=f"MMLU-{RUN_NAME}",
    project=PROJECT,
    config={**CONFIG, **GEN, "eval_dataset": "mmlu"}
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

define_wandb_chart_only_metrics()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260616_173451-9ayzzjrx
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run MMLU-Baseline-Gemma-4-E4B-IT
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/9ayzzjrx


✅ W&B run: MMLU-Baseline-Gemma-4-E4B-IT  —  https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/9ayzzjrx


## 10. MMLU Evaluation

We evaluate the single-agent baseline over the fixed 128-question subset, repeating each question under 3 generation seeds. For each `(question, seed)` we record correctness, latency, token counts and peak VRAM.

In [11]:
import warnings
warnings.filterwarnings("ignore") # to ignore transformer warnings that can clutter the output during evaluation

import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from utils.data import format_mmlu_prompt
from utils.evaluation import normalize_mmlu_answer, get_ground_truth_letter, mmlu_match
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

# Fix the evaluation subset once — all seed runs evaluate the same questions
random.seed(wandb.config.data_seed)
test_sample = random.sample(list(mmlu_dataset), sample_size)

# ── Baseline matrices: rows = problems, cols = seeds ───────────────────
baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_input_tok_matrix   = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_output_tok_matrix  = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_gpu_peak_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=float)

# ── Single unified results table accumulated across ALL seeds ─────────────
results_columns = [
    "model_id", "dataset", "protocol",
    "problem_id", "seed",
    "task",
    "output",
    "ground_truth", "final_output",
    "latency_s",
    "output_tokens", "total_tokens",
    "gpu_peak_mb",
    "throughput_tok_s",
]
results_rows = []

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    # Rebuild LLM and the baseline chain with the current generation seed
    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens,
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    baseline_chain = mmlu_template | llm | parser

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"MMLU seed={gen_seed} ({sample_size} samples)")):
        question        = item['question']
        choices         = item['choices']
        true_answer_idx = item['answer']
        true_choice     = get_ground_truth_letter(true_answer_idx)
        task            = format_mmlu_prompt(question, choices)

        output = None
        try:
            output            = run_single_agent_baseline(baseline_chain, task, task_label='mmlu_sample')
            baseline_response = output['baseline']
            baseline_choice   = normalize_mmlu_answer(baseline_response)
        except Exception as e:
            print(f"Baseline failed: {e}")
            baseline_choice   = None
            baseline_response = str(e)
            if output is None:
                output = {
                    'baseline': baseline_response,
                    't_base': 0.0,
                    'base_in_tok': 0, 'base_out_tok': 0, 'base_total_tok': 0,
                    'base_peak_mb': 0.0, 'base_throughput': 0.0,
                }

        baseline_correct = mmlu_match(baseline_choice, true_choice)

        # ── Fill baseline matrices ────────────────────────────────────
        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']
        baseline_input_tok_matrix[item_idx, run_idx]   = output['base_in_tok']
        baseline_output_tok_matrix[item_idx, run_idx]  = output['base_out_tok']
        baseline_gpu_peak_matrix[item_idx, run_idx]    = output['base_peak_mb']

        log_metrics({
            "mmlu_sample/accuracy": int(baseline_correct),
        })

        # Accumulate one row for the unified results table (problem_id, seed first)
        results_rows.append([
            CONFIG['model_id'], "mmlu", CONFIG['protocol'],
            item_idx, gen_seed,
            task,
            output['baseline'][:2000],
            true_choice, baseline_choice,
            round(output['t_base'], 2),
            output['base_out_tok'], output['base_total_tok'],
            round(output['base_peak_mb'], 2),
            round(output['base_throughput'], 4),
        ])

# ── Log the unified results table ONCE, sorted by problem_id then seed ─────
results_rows.sort(key=lambda r: (r[3], r[4]))
results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({f"results_comparison_MMLU": results_table})


  Seed run 1/3  —  generation_seed=42


MMLU seed=42 (128 samples):   0%|          | 0/128 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 SINGLE-AGENT BASELINE  —  mmlu_sample


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  ⏱  mmlu_sample/baseline: 37.3s  |  1259 chars  |  in=75 out=247 tok  |  peak=9107 MB  |  6.6 tok/s

📊 SINGLE-AGENT BASELINE  —  mmlu_sample


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  mmlu_sample/baseline: 33.3s  |  599 chars  |  in=79 out=219 tok  |  peak=9108 MB  |  6.6 tok/s

📊 SINGLE-AGENT BASELINE  —  mmlu_sample


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  mmlu_sample/baseline: 74.2s  |  2126 chars  |  in=110 out=496 tok  |  peak=9113 MB  |  6.7 tok/s

📊 SINGLE-AGENT BASELINE  —  mmlu_sample
  ⏱  mmlu_sample/baseline: 82.5s  |  2245 chars  |  in=92 out=551 tok  |  peak=9115 MB  |  6.7 tok/s


## 11. MMLU — Results, Statistics & Finish W&B Run

We compute cluster-robust statistics for the baseline (accuracy, latency, output/total tokens, peak VRAM), push them and the results table to W&B, persist the matrices as a W&B artifact, and close the MMLU run.

In [12]:
import tempfile
import wandb
import numpy as np
from utils.logging_utils import finish_wandb, log_stat_summary
from utils.statistics import (
    accuracy_statistics,
    time_statistics,
    token_statistics,
    vram_statistics,
    print_accuracy_stats_report,
    print_time_report,
    print_token_report,
    print_vram_report,
)

DATASET = "MMLU"

# ── Derived matrix ───────────────────────────────────────────────────────
baseline_total_tok_matrix = baseline_input_tok_matrix + baseline_output_tok_matrix

# ── Accuracy ─────────────────────────────────────────────────────────────
baseline_acc_stats = accuracy_statistics(baseline_correctness_matrix)
print_accuracy_stats_report(f"Baseline Accuracy ({DATASET})", baseline_acc_stats)

# ── Latency ──────────────────────────────────────────────────────────────
baseline_time_stats = time_statistics(baseline_time_matrix)
print_time_report(f"Baseline Latency ({DATASET})", baseline_time_stats)

# ── Output token counts ───────────────────────────────────────────────────
baseline_out_tok_stats = token_statistics(baseline_output_tok_matrix, max_tokens=wandb.config.max_new_tokens)
print_token_report(f"Baseline Output Tokens ({DATASET})", baseline_out_tok_stats)

# ── Total token counts (input + output) ─────────────────────────────────
baseline_total_tok_stats = token_statistics(baseline_total_tok_matrix)
print_token_report(f"Baseline Total Tokens ({DATASET})", baseline_total_tok_stats)

# ── Peak VRAM ─────────────────────────────────────────────────────────────
baseline_vram_stats = vram_statistics(baseline_gpu_peak_matrix)
print_vram_report(f"Baseline Peak VRAM ({DATASET})", baseline_vram_stats)

# ── W&B summary ──────────────────────────────────────────────────────────
wandb.summary["n_seeds"]           = len(generation_seeds)
wandb.summary["n_problems"]        = sample_size
wandb.summary["total_experiments"] = sample_size * len(generation_seeds)

# Baseline: accuracy statistics
wandb.summary["accuracy"] = round(baseline_acc_stats["p_hat"], 6)
log_stat_summary("accuracy_stats", baseline_acc_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "wilson_ci_lower", "wilson_ci_upper", "wilson_n_eff", "wilson_icc"])

# Baseline: latency statistics
wandb.summary["latency"] = round(baseline_time_stats["mean"], 2)
log_stat_summary("latency_stats", baseline_time_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "flat_cv"])

# Baseline: output token statistics
wandb.summary["output_tokens"] = round(baseline_out_tok_stats["mean"], 1)
log_stat_summary("output_tokens_stats", baseline_out_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])

# Baseline: total token statistics
wandb.summary["total_tokens"] = round(baseline_total_tok_stats["mean"], 1)
log_stat_summary("total_tokens_stats", baseline_total_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])

# Baseline: peak VRAM statistics
wandb.summary["mean_vram_peak"] = round(baseline_vram_stats["mean_peak"], 0)
wandb.summary["max_vram_peak"]  = round(baseline_vram_stats["peak_max"], 0)
log_stat_summary("vram_peak_stats", baseline_vram_stats,
    keys=["peak_ci_lower", "peak_ci_upper", "between_problem_peak_std"])


# ── Upload all matrices as W&B artifact ──────────────────────────────────
artifact = wandb.Artifact(
    f"evaluation-matrices-Gemma-4-E4B-it-Baseline-{DATASET}",
    type="evaluation-data",
    description=(
        f"Single-agent baseline evaluation matrices ({len(generation_seeds)} x {sample_size}). {DATASET} evaluation with {CONFIG['model_id']}."
        "Includes: correctness, execution times, input/output/total token counts, "
        "GPU peak VRAM (MB)."
    ),
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        baseline_correctness=baseline_correctness_matrix,
        baseline_times=baseline_time_matrix,
        baseline_input_tokens=baseline_input_tok_matrix,
        baseline_output_tokens=baseline_output_tok_matrix,
        baseline_total_tokens=baseline_total_tok_matrix,
        baseline_gpu_peak=baseline_gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print(f"✅ All evaluation matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished. View your dashboard at the URL above.")


  Baseline Accuracy (MMLU)
  Accuracy p̂                  69.53%
  95% Cluster-t CI         [62.51, 76.55]%
  Between-problem std        40.1283%
  ── Supplementary: Wilson ──
  95% Wilson CI              [62.21, 75.98]%
  DEFF                       168.401
  ICC                          0.640
  ── Sample ──
  Problems x Seeds           128 x 3  (n=384)


  Baseline Latency (MMLU)
  Mean (s)                     72.5118
  Std (s)                      30.3926
  CV                            0.4191
  95% BCa CI (s)             [67.8191, 77.6814]  [cluster_bca]
  Between-problem std (s)      28.7553
  ── Percentiles (descriptive) ──
  Median / P50 (s)             64.2579
  P25 (s)                      51.0915
  P75 (s)                      88.4451
  IQR (s)                      37.3536
  P95 (s)                     145.3257
  ── Sample ──
  Problems x Seeds           128 x 3  (n=384)


  Baseline Output Tokens (MMLU)
  Mean tok                        486.1328
  Std tok                    

wandb: uploading artifact run-9ayzzjrx-results_comparison_MMLU; updating run metadata; uploading artifact evaluation-matrices-Gemma-4-E4B-it-Baseline-MMLU
wandb: uploading artifact evaluation-matrices-Gemma-4-E4B-it-Baseline-MMLU
wandb: uploading media/table/results_comparison_MMLU_768_0c061b657b8d1f77a111.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       mmlu_sample/accuracy ▁▁██▁█████▁▁██▁██▁▁▁███▁████████▁████▁▁█
wandb:    mmlu_sample/gpu_peak_mb ▄▁▂▂▂▂▂▂▃▃▁▂▃▂▆▁▁▃▃▁▂█▇█▁▃▆▂▃▂▃▅▂▃▁▂▁▂▃▅
wandb:      mmlu_sample/latency_s ▄▂▂▁▂▆▂▃▃▁▄▂▂▃▄█▂▄▂▅▃▃▂█▃▅▃▁▃▇▃▃▅▄▂▂▂█▆▁
wandb:  mmlu_sample/output_tokens ▄▄▄▂▃▃▂▄▃▂█▂▆▂▄▂▂▆▅▁▃█▂▂▄▁▇▃▃▂▅▄▁▄▄▁▄▄▁▄
wandb: mmlu_sample/throughput_tps ▁▅▃▆▆▄▃▅▂▄▅▂▃▅▄▂▅▄▂▃▃▆▇▆▆▅▄▃▆▇▅▁▃▅▃▃▃▃█▄
wandb:   mmlu_sample/total_tokens ▃▂▂▅▃▃▂▆▂▂▂▃▂▃▂▂▄▅▆▂▃▃▅▂▂▁█▂▁▃▂▃▅▂▂▂▁▄▃▂
wandb: 
wandb: Run summary:
wandb:                           accuracy 0.69531
wandb: accuracy_stats/between_problem_std 0.40128


W&B run finished. View your dashboard at the URL above.


## 12. Analysis & Next Steps

### What W&B gives you (this run)
- **Correctness / accuracy** — per-sample and aggregate baseline accuracy with a 95% CI, for MMLU.
- **Latency, token and VRAM panels** — per-sample `mmlu_sample/baseline_*` metrics and their cluster-robust summaries.
- **Results table** — every sample's prompt, output, predicted vs. ground-truth letter, latency, tokens, peak VRAM and throughput.
- **Artifact** — `evaluation_matrices.npz` with the baseline correctness / latency / token / VRAM matrices (N×m) for MMLU, for downstream statistical analysis.

### Role of this notebook
This is the **single-agent baseline** for MMLU with a SOTA SLM (**Gemma-4-E4B-it**, run text-only from its multimodal checkpoint). It runs only the MMLU evaluation — the GSM8K counterpart is `v2.4.0` — because Kaggle's 12-hour session limit does not reliably fit both evaluations for this model in one run. Combined with the GSM8K run logged under `v2.4.0`, the two W&B runs together form the complete single-agent baseline for this model. **Overhead ratios/deltas are intentionally not computed here** — the MAS notebooks will load these matrices and compute MAS-vs-baseline overhead against them.

### Next steps
1. Build the multi-agent system from instances of this same model and compare accuracy and cost (latency + total tokens + VRAM) against both baselines (OBJ-1,2,3).